In [1]:
from src.envs.generators import FlatGraphs
from src.envs.encoders import Graph
from types import SimpleNamespace
from pyzx import draw
import numpy as np

params = SimpleNamespace(
    min_qubits = 5,
    max_qubits = 5,
    qubit_step = 1,
    min_depth = 15,
    max_depth = 15,
    depth_step = 1,
    circuit_type = None,

    log_qubits_depth = True,
    precise_vocab = False,
    max_int = 300,
    max_nodes = 300,
)

rng = np.random.default_rng(42)


In [2]:
from src.trainer import Trainer
from src.evaluator import Evaluator
from src.model import check_model_params, build_modules
from src.envs import ENVS, build_env
from src.dataset import EnvDataset

import torch
import os

dump_path = "/home/ryeung/dumped/zx_flat/zze7uigbb1"
checkpoint_path = os.path.join(dump_path, "checkpoint.pth")
data = torch.load(checkpoint_path, map_location=torch.device('cpu'))

params = SimpleNamespace(**data['params'])
task = params.tasks = "simplify"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = build_env(params)
modules = build_modules(env, params)
trainer = Trainer(modules, env, params)
evaluator = Evaluator(trainer)

env_dataset = EnvDataset(
    env,
    task,
    train=False,
    params=params,
    path=None,
    size=1000,
    type="valid",
)

Reloading checkpoint from /home/ryeung/dumped/zx_flat/zze7uigbb1/checkpoint.pth ...
Reloading checkpoint optimizer ...
Reloading gradient scaler ...
Checkpoint reloaded. Resuming at epoch 48 / iteration 225024 ...


In [3]:
# (x1, len1), (x2, len2), _ = trainer.get_batch(task="simplify")[2]

In [4]:
encoder = trainer.modules["encoder"]
decoder = trainer.modules["decoder"]

In [5]:
gen = FlatGraphs(params)
g = gen.generate(rng)

enc = Graph(params)

In [6]:
tokens = env.encoder.encode(g[0])
ids = torch.tensor([env.word2id[t] for t in tokens], device=device)
x, x_len = env_dataset.batch_sequences(ids.unsqueeze(0), env.pad_index, env.eos_index)
x, x_len = x.to(device), x_len.to(device)


In [7]:
encoded = encoder("fwd", x=x, lengths=x_len, causal=False)

In [8]:
encoded

tensor([[[ 0.5512, -0.3443, -0.0630,  ..., -0.0921, -0.3384, -0.6050]],

        [[-0.0574, -0.0668,  0.0216,  ...,  0.2812, -0.0083,  0.0232]],

        [[ 0.2000,  0.2679, -0.1540,  ..., -0.3096, -0.5729,  0.4770]],

        ...,

        [[-0.0568, -0.0665,  0.0204,  ...,  0.2788, -0.0051,  0.0158]],

        [[-0.0785, -0.0544,  0.0144,  ...,  0.2738, -0.0071,  0.0249]],

        [[-0.0635, -0.0621, -0.0129,  ...,  0.1730, -0.0601,  0.0210]]],
       device='cuda:0', grad_fn=<TransposeBackward0>)

In [9]:
params.max_output_len

300

In [10]:
encoded = encoder("fwd", x=x, lengths=x_len, causal=False)
# generate
generated, _ = decoder.generate(
    encoded.transpose(0, 1),
    x_len,
    # max_len=params.max_output_len + 2,
    max_len=1000,
)
generated=generated.transpose(0, 1)


In [11]:
print(', '.join(tokens))

90, 85, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 2, 2, 2, 1, 4, 2, 0, 1, 4, 2, 4, 1, 0, 2, 0, 1, 2, 2, 0, 1, 6, 2, 0, 1, 4, 2, 0, 1, 0, 2, 2, 1, 4, 2, 0, 1, 0, 2, 6, 1, 0, 2, 0, 1, 6, 2, 0, 1, 0, 2, 2, 1, 4, 2, 6, 1, 0, 2, 0, 1, 2, 2, 0, 1, 0, 2, 0, 1, 2, 2, 0, 1, 4, 2, 4, 1, 0, 2, 4, 1, 2, 2, 6, 1, 0, 2, 6, 1, 2, 2, 0, 1, 4, 2, 4, 1, 0, 2, 6, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, N0, N10, 1, N1, N11, 1, N2, N12, 1, N3, N13, 1, N4, N14, 1, N5, N15, 1, N6, N16, 1, N7, N17, 1, N8, N18, 1, N9, N19, 1, N10, N21, 1, N11, N24, 1, N12, N28, 1, N13, N42, 2, N14, N36, 1, N15, N20, 1, N16, N34, 1, N17, N33, 1, N18, N23, 1, N19, N22, 1, N20, N21, 1, N20, N37, 1, N21, N26, 1, N22, N25, 1, N23, N27, 1, N24, N30, 1, N25, N35, 1, N26, N27, 1, N26, N29, 2, N27, N39, 2, N28, N31, 1, N29, N40, 1, N30, N31, 2, N30, N45, 1, N31, N32, 1, N32,

In [12]:
print(', '.join([env.id2word[int(x)] for x in generated[0]]))

<s>, 70, 65, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 2, 1, 2, 1, 6, 1, 0, 1, 2, 1, 0, 1, 0, 1, 0, 1, 4, 2, 2, 2, 0, 2, 6, 1, 0, 2, 0, 2, 0, 1, 0, 2, 0, 1, 4, 2, 6, 2, 6, 1, 0, 2, 0, 1, 0, 2, 2, 1, 0, 2, 0, 1, 2, 2, 0, 1, 0, 2, 0, 1, 2, 1, 4, 2, 4, 1, 0, 2, 4, 1, 2, 2, 6, 1, 0, 2, 6, 1, 2, 2, 0, 1, 4, 2, 4, 1, 0, 2, 6, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, N0, N10, 1, N1, N11, 1, N2, N12, 1, N3, N13, 1, N4, N14, 1, N5, N15, 1, N6, N16, 1, N7, N17, 1, N8, N18, 1, N9, N19, 1, N10, N20, 1, N11, N25, 2, N11, N34, 2, N12, N25, 1, N13, N63, 2, N14, N28, 1, N15, N20, 1, N15, N29, 2, N16, N32, 1, N17, N27, 1, N18, N21, 1, N19, N22, 1, N20, N23, 1, N21, N23, 2, N21, N30, 2, N22, N48, 1, N23, N24, 1, N24, N31, 1, N25, N26, 1, N26, N30, 1, N26, N36, 1, N27, N39, 2, N28, N37, 1, N29, N41, 2, N30, N44, 2, N31, N53, 1, N32, N33, 2, N33, N38, 1, N34, N35, 2, N35, N46, 2, N36, N45, 1, N37, N40, 1, N38, N42, 1, N39, N40, 1, N3

In [13]:
new_g = env.encoder.decode([env.id2word[int(x)] for x in generated[0, 1:-1]])
draw(new_g)


In [14]:
from pyzx import full_reduce
combined = g[0] + new_g.adjoint()
full_reduce(combined)

draw(combined)

In [15]:
draw(g[0])

In [16]:
generated

tensor([[   0,  684,  628,   15,   15,   15,   15,   15,   15,   15,   15,   15,
           15,   15,   15,   15,   15,   15,   15,   15,   15,   15,   15,   16,
           15,   16,  128,   16,  128,   16,  572,   16,   15,   16,  128,   16,
           15,   16,   15,   16,   15,   16,  350,  128,  128,  128,   15,  128,
          572,   16,   15,  128,   15,  128,   15,   16,   15,  128,   15,   16,
          350,  128,  572,  128,  572,   16,   15,  128,   15,   16,   15,  128,
          128,   16,   15,  128,   15,   16,  128,  128,   15,   16,   15,  128,
           15,   16,  128,   16,  350,  128,  350,   16,   15,  128,  350,   16,
          128,  128,  572,   16,   15,  128,  572,   16,  128,  128,   15,   16,
          350,  128,  350,   16,   15,  128,  572,   16,   15,   16,   15,   16,
           15,   16,   15,   15,   15,   15,   15,   15,   15,   15,   15,   15,
           15,   15,   15,   15,   15,   15,   15,   15,   15,   15,   15,   16,
         1016, 1018,   16, 1

In [17]:
tokens

['90',
 '85',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '2',
 '2',
 '2',
 '1',
 '4',
 '2',
 '0',
 '1',
 '4',
 '2',
 '4',
 '1',
 '0',
 '2',
 '0',
 '1',
 '2',
 '2',
 '0',
 '1',
 '6',
 '2',
 '0',
 '1',
 '4',
 '2',
 '0',
 '1',
 '0',
 '2',
 '2',
 '1',
 '4',
 '2',
 '0',
 '1',
 '0',
 '2',
 '6',
 '1',
 '0',
 '2',
 '0',
 '1',
 '6',
 '2',
 '0',
 '1',
 '0',
 '2',
 '2',
 '1',
 '4',
 '2',
 '6',
 '1',
 '0',
 '2',
 '0',
 '1',
 '2',
 '2',
 '0',
 '1',
 '0',
 '2',
 '0',
 '1',
 '2',
 '2',
 '0',
 '1',
 '4',
 '2',
 '4',
 '1',
 '0',
 '2',
 '4',
 '1',
 '2',
 '2',
 '6',
 '1',
 '0',
 '2',
 '6',
 '1',
 '2',
 '2',
 '0',
 '1',
 '4',
 '2',
 '4',
 '1',
 '0',
 '2',
 '6',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '1',
 '0',
 '0',
 '0',
 '0',
 '0',
 '

In [18]:
from pyzx import draw

In [19]:
def do_a_step(g):
    tokens = env.encoder.encode(g)
    ids = torch.tensor([env.word2id[t] for t in tokens], device=device)
    x, x_len = env_dataset.batch_sequences(ids.unsqueeze(0), env.pad_index, env.eos_index)
    x, x_len = x.to(device), x_len.to(device)

    encoded = encoder("fwd", x=x, lengths=x_len, causal=False)
    generated, _ = decoder.generate(
        encoded.transpose(0, 1),
        x_len,
        max_len=600,
    )
    generated=generated.transpose(0, 1)
    new_g = env.encoder.decode([env.id2word[int(x)] for x in generated[0, 1:-1]])
    return new_g

In [20]:
def do_the_steps(g):
    gs = []
    while g is not None:
        gs.append(g)
        g = do_a_step(g)
        print('.', end='')
    print()

    for i, (g1, g2) in enumerate(zip(gs[:-1], gs[1:])):
        combined = g1 + g2.adjoint()
        full_reduce(combined)
        print(f'g{i} == g{i+1}:', combined.is_id())

    for i, g in enumerate(gs):
        draw(g)
    return gs

In [24]:
gen.new_graph(rng)
g = gen.generate(rng)[0]
gs = do_the_steps(g)

g0 == g1: True
g1 == g2: True
g2 == g3: True


In [22]:
g_last = gs[-1].copy()
g_last_copy = g_last.copy()
full_reduce(g_last_copy)
print(g_last.stats())
print(g_last_copy.stats())

Graph(90 vertices, 91 edges)
degree distribution: 
1: 20
2: 48
3: 22

Graph(36 vertices, 40 edges)
degree distribution: 
1: 20
2: 3
3: 6
4: 4
5: 1
7: 1
8: 1



In [23]:
for g in gs:
    print(g.stats())

Graph(90 vertices, 91 edges)
degree distribution: 
1: 20
2: 48
3: 22

